# 01. Nifty 500 Data Coverage & Ingestion Audit

This notebook audits constituent ingestion, sector categorization, and financial
statement completeness across the Nifty 500 universe.

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "data").exists():
    project_root = project_root.parent

interim_dir = project_root / "data" / "interim"
ref_dir = project_root / "data" / "reference"

## 1. Constituent Ingestion & Universe Definition

The official NSE Nifty 500 constituent listing contains 501 rows:
- 496 Equity (EQ) series
- 2 SME/Trade-for-Trade (BE) series
- 3 Real Estate Investment Trusts (RR), excluded as non-equity trusts
- 98 Financial Services companies, excluded due to distinct accounting structures
This yields a target universe of 400 non-financial operating companies.

In [ ]:
constituents_path = ref_dir / "nifty500_constituents.csv"
if constituents_path.exists():
    df_const = pd.read_csv(constituents_path)
    print(f"Total constituent rows: {len(df_const)}")
    print("Series distribution:")
    print(df_const["series"].value_counts())

## 2. Statement Completeness Matrix

Reviewing line-item coverage across Balance Sheet, Income Statement, and Cash Flow
Statement from data/interim/coverage_report.csv.

In [ ]:
coverage_path = interim_dir / "coverage_report.csv"
if coverage_path.exists():
    df_cov = pd.read_csv(coverage_path)
    print(f"Total mapped standard fields: {len(df_cov)}")
    display_cols = ["statement", "standard_field", "pct_ge_1yr", "pct_ge_3yr", "pct_ge_4yr"]
    print(df_cov[display_cols].to_string(index=False))

## 3. Unmapped Line Item Inspection

Analysis of raw Yahoo Finance row labels that were not mapped to standard fields.

In [ ]:
unmapped_path = interim_dir / "unmapped_labels.csv"
if unmapped_path.exists():
    df_unmapped = pd.read_csv(unmapped_path)
    print(f"Total unmapped line occurrences: {len(df_unmapped)}")
    top_unmapped = df_unmapped["raw_label"].value_counts().head(10)
    print("Top 10 unmapped labels by frequency:")
    print(top_unmapped)

## 4. Ingestion Summary

Of the 400 target non-financial equities, 399 companies were fetched (99.8%).
The sole missing ticker, DUMMYHEG.NS, represents a temporary demerger placeholder.
All primary financial statement line items achieve >95% coverage across 3+ fiscal years.